# Урок 23 — Stack, Queue, Deque: Архітектура Потоку Даних

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Перестаньте думати про структури даних як про "контейнери" або коробки для зберігання змінних.  
Натомість думайте про них як про **механізми маршрутизації**, які диктують *коли* і *в якому порядку* ваші дані отримають свою чергу на обробку.

Коли дані потрапляють у вашу систему, обрана структура визначає точно **в якому порядку кожен елемент буде оброблений**.  
Це не просто деталь реалізації — це архітектурне рішення, яке визначає поведінку всього алгоритму.

| Частина | Структура | Патерн | Застосування |
|---------|-----------|--------|--------------|
| 1 | `list` як Stack | LIFO — «Переривання» | Рекурсія, DFS, Undo |
| 2 | `deque` як Queue | FIFO — «Справедливість» | BFS, Task Pipelines |
| 3 | `deque` | Двосторонній потік | Sliding Window, Circular Buffer |
| 4 | `queue.Queue` | Thread-safe | Конкурентне програмування |
| 5 | Explicit Stack | Без рекурсії | DFS без `RecursionError` |
| 6 | Питання для співбесіди + Завдання | — | — |

---

---

## Частина 1: Stack — Патерн «Переривання» (LIFO)

---

### Ментальна модель

Stack (Стек) — це структура **LIFO: Last-In, First-Out** (Останній прийшов — перший вийшов).

Поведінково стек представляє **переривання**.  
Коли надходить нова задача, вона вимагає негайної уваги — ви призупиняєте поточну роботу.  
Ви повертаєтесь до старіших задач лише тоді, коли нова задача повністю вирішена.

**Аналогія:** стос тарілок у кафетерії або стопка пошти на столі.  
Ви читаєте звіт, і шеф кидає зверху термінове повідомлення — ви зупиняєтеся, обробляєте його, і тільки тоді повертаєтеся до звіту.

**Реальні застосування:**
- Call Stack (Рекурсія) — як ваш процесор виконує функції
- Depth-First Search (DFS) — обхід дерев і графів
- Кнопка Undo в текстових редакторах
- Перевірка балансу дужок у парсерах

### Python-реалізація стека: `list`

У Python не потрібно імпортувати спеціальний клас для стека.  
Вбудований `list` оптимізований для стекової поведінки:

| Операція | Метод | Складність |
|----------|-------|------------|
| Push (покласти зверху) | `list.append(item)` | **O(1)** |
| Pop (зняти зверху) | `list.pop()` | **O(1)** |
| Peek (подивитися зверху) | `stack[-1]` | **O(1)** |

Оскільки всі операції відбуваються в **кінці масиву**, Python не перерозподіляє пам'ять —  
це амортизовані O(1) операції.

In [ ]:
# === STACK: реалізація на базі list ===

stack = []

# Push: кладемо задачі на стос (O(1))
stack.append("Задача A: написати тести")
stack.append("Задача B: виправити баг")
stack.append("Задача C: ТЕРМІНОВО — падає прод!")

print("Стек задач (знизу → верх):")
for i, task in enumerate(stack):
    print(f"  [{i}] {task}")

print()
print(f"Зараз вгорі (peek): {stack[-1]}")  # Peek — не видаляємо

print()
print("Обробка задач (LIFO — остання прийшла, перша вийшла):")
while stack:
    task = stack.pop()  # Pop: завжди знімаємо з кінця (O(1))
    print(f"  → Обробляємо: {task}")

### Архітектурний патерн: Call Stack і Рекурсія

**Call Stack** — це те, як CPython керує виконанням функцій.  
Кожен виклик функції **«пушить»** новий стековий фрейм (stack frame) з:
1. **Адресою повернення** — рядком коду, куди повернутися після завершення
2. **Локальними змінними** — ізольоване оточення функції

Коли функція виконує `return`, фрейм **«попається»** зі стека, локальні змінні знищуються.  

> **Небезпека:** Кожен рекурсивний виклик споживає пам'ять для нового фрейму.  
> Python обмежує глибину рекурсії (за замовчуванням ~1000 фреймів).  
> Перевищення викидає `RecursionError`.

In [ ]:
import sys

# Перевіряємо поточний ліміт рекурсії
print(f"Ліміт рекурсії Python: {sys.getrecursionlimit()} фреймів")


def factorial(n: int) -> int:
    """Обчислення факторіалу через рекурсію.
    
    Кожен виклик fact(n) кладе фрейм на call stack;
    base case (n <= 1) зупиняє занурення і починає розгортання.
    """
    # Базовий випадок: зупиняємо рекурсію, починаємо розгортання стека
    if n <= 1:
        print(f"  BASE CASE → fact({n}) = 1, починаємо розгортання")
        return 1
    
    print(f"  PUSH → fact({n}) чекає на fact({n-1})")
    result = n * factorial(n - 1)  # Рекурсивний виклик — новий фрейм на стек
    print(f"  POP  → fact({n}) = {result}")
    return result


print("\nВиклик factorial(4):")
print(f"\nРезультат: {factorial(4)}")

### Патерн: Перевірка балансу дужок

Класичне застосування стека — перевірка, чи всі дужки у виразі закриті.  
Це основа будь-якого парсера (компілятора, XML/JSON-парсера).

In [ ]:
def is_balanced(expression: str) -> bool:
    """Перевіряє, чи всі дужки у виразі збалансовані."""
    stack = []  # Стек для відслідковування відкритих дужок
    
    # Відповідності: кожна закриваюча дужка → її відкриваюча пара
    closing_to_opening = {')': '(', ']': '[', '}': '{'}
    
    for char in expression:
        if char in '([{':          # Відкриваюча дужка — кладемо на стек
            stack.append(char)
        elif char in ')]}':        # Закриваюча дужка — перевіряємо відповідність
            if not stack:          # Стек порожній — нема чому закриватися
                return False
            if stack[-1] != closing_to_opening[char]:  # Пара не збігається
                return False
            stack.pop()            # Знімаємо пару зі стека
    
    return len(stack) == 0         # Якщо стек порожній — всі дужки закриті


test_cases = [
    ("({[]})",           True),
    ("function(a[0])",   True),
    ("({[})",            False),   # Неправильна пара
    ("(((",              False),   # Незакриті дужки
    ("",                 True),    # Порожній рядок — збалансований
]

for expr, expected in test_cases:
    result = is_balanced(expr)
    status = "✓" if result == expected else "✗"
    print(f"  {status}  is_balanced({expr!r:20}) = {result}")

---

## Частина 2: Queue — Патерн «Справедливість» (FIFO)

---

### Ментальна модель

Queue (Черга) — це структура **FIFO: First-In, First-Out** (Перший прийшов — перший вийшов).

Поведінково черга представляє **справедливість і буферизацію**.  
Дані обробляються в точному хронологічному порядку їх надходження.  
Черга поглинає піки трафіку, гарантуючи рівномірне навантаження на споживача.

**Аналогія:** касова черга у супермаркеті або черга принтера.  
Хто прийшов першим — обслуговується першим. Нові клієнти встають у кінець черги.

### Архітектурна пастка: чому `list` — токсичний для черги

```python
# НЕПРАВИЛЬНО — не робіть так!
queue = []
queue.append("item")   # O(1) — нормально
queue.pop(0)            # O(n) — КАТАСТРОФА!
```

Список Python — це безперервний масив у пам'яті.  
Вилучення першого елемента (`pop(0)`) лишає "дірку" на початку — Python мусить **фізично зсунути** всі наступні елементи на одну позицію вліво.  

| Розмір черги | Операцій при pop(0) |
|--------------|--------------------|
| 1 000 | 1 000 зсувів |
| 1 000 000 | 1 000 000 зсувів |

У циклі це деградує до **O(n²)** — система зупиниться.

In [ ]:
import timeit
from collections import deque

N = 10_000  # Кількість елементів

# Вимірюємо list.pop(0) — O(n) операція
time_list = timeit.timeit(
    stmt='q.pop(0)',
    setup=f'q = list(range({N}))',
    number=N // 2  # Виконуємо N/2 pop(0)
)

# Вимірюємо deque.popleft() — O(1) операція
time_deque = timeit.timeit(
    stmt='q.popleft()',
    setup=f'from collections import deque; q = deque(range({N}))',
    number=N // 2  # Виконуємо N/2 popleft()
)

print(f"list.pop(0)    : {time_list:.4f} сек — O(n), зсуває пам'ять")
print(f"deque.popleft(): {time_deque:.6f} сек — O(1), просто рухає вказівник")
print(f"\ndeque швидший у {time_list / time_deque:.1f}x разів")

### Правильна реалізація черги: `collections.deque`

**`deque`** (Double-Ended Queue) реалізований у C як **двоспрямований зв'язний список**.  
Структура зберігає вказівники на початок і кінець — тому операції на обох кінцях суворо O(1).

| Операція | Метод | Складність |
|----------|-------|------------|
| Enqueue (у кінець) | `deque.append(item)` | **O(1)** |
| Dequeue (з початку) | `deque.popleft()` | **O(1)** |
| Peek (перший елемент) | `queue[0]` | **O(1)** |
| Розмір | `len(queue)` | **O(1)** |

In [ ]:
from collections import deque

# Створюємо чергу HTTP-запитів до сервера
request_queue: deque[str] = deque()

# Enqueue: клієнти надходять у систему (FIFO — в кінець)
for i in range(1, 6):
    request_queue.append(f"Request-{i:03d}")
    print(f"  +  Додано: Request-{i:03d} | Черга: {len(request_queue)}")

print()
print(f"Очікує обробки: {len(request_queue)} запитів")
print(f"Перший у черзі (peek): {request_queue[0]}")

print()
print("Обробка запитів (FIFO — перший прийшов, перший вийшов):")
while request_queue:
    current = request_queue.popleft()  # Dequeue: завжди з початку (O(1))
    print(f"  → Обробляємо: {current} | Залишилось: {len(request_queue)}")

### Архітектурний патерн: Breadth-First Search (BFS)

**BFS** — Пошук в ширину — досліджує граф **шар за шаром** (радіальне розширення).  
Він використовує FIFO-чергу, щоб гарантувати обробку ближніх вузлів перед дальніми.

**Архітектурна властивість:** завдяки FIFO BFS **завжди знаходить найкоротший шлях**  
за кількістю ребер у незваженому графі.

Порівняння зі Stack/DFS:
- **Stack (DFS):** «Нервово» пірнає глибоко в одну гілку, ігноруючи інші
- **Queue (BFS):** «Обережно» досліджує рівень за рівнем, не поспішаючи

In [ ]:
from collections import deque
from typing import Optional

# Граф соціальної мережі: хто з ким дружить
social_network = {
    "Alice":   ["Bob", "Carol"],
    "Bob":     ["Alice", "Dave", "Eve"],
    "Carol":   ["Alice", "Frank"],
    "Dave":    ["Bob"],
    "Eve":     ["Bob", "Grace"],
    "Frank":   ["Carol"],
    "Grace":   ["Eve"],
}

def bfs_shortest_path(
    graph: dict[str, list[str]],
    start: str,
    target: str
) -> Optional[list[str]]:
    """Знаходить найкоротший шлях між двома вузлами через BFS."""
    # Ініціалізуємо FIFO-чергу зі стартового вузла
    # Зберігаємо шлях — список вузлів, щоб відстежити маршрут
    queue: deque[list[str]] = deque([[start]])
    
    # Множина відвіданих вузлів — уникаємо нескінченних циклів
    visited: set[str] = {start}
    
    while queue:
        path = queue.popleft()       # Беремо найстаріший (найближчий) шлях
        current_node = path[-1]      # Поточний вузол — останній у шляху
        
        if current_node == target:   # Знайшли ціль — повертаємо шлях
            return path
        
        # Додаємо всіх невідвіданих сусідів у КІНЕЦЬ черги (FIFO)
        for neighbor in graph.get(current_node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])  # Новий шлях = старий + сусід
    
    return None  # Шлях не знайдено


# Пошук найкоротшого ланцюжка знайомств
path = bfs_shortest_path(social_network, "Alice", "Grace")
print(f"Найкоротший шлях Alice → Grace:")
print(f"  {' → '.join(path)}  ({len(path)-1} кроки)")

path2 = bfs_shortest_path(social_network, "Alice", "Frank")
print(f"\nНайкоротший шлях Alice → Frank:")
print(f"  {' → '.join(path2)}  ({len(path2)-1} кроки)")

---

## Частина 3: Deque — Двосторонній Потік Даних

---

### Що таке `deque`?

**`deque`** (вимовляється «дек») — Double-Ended Queue, двостороння черга.  
Це **узагальнення** лінійних структур: дозволяє вставляти й вилучати елементи **з обох кінців**.

| Операція | Метод | Кінець | Складність |
|----------|-------|--------|------------|
| Append right | `deque.append(x)` | Правий | **O(1)** |
| Append left | `deque.appendleft(x)` | Лівий | **O(1)** |
| Pop right | `deque.pop()` | Правий | **O(1)** |
| Pop left | `deque.popleft()` | Лівий | **O(1)** |
| Rotate | `deque.rotate(n)` | Обидва | **O(k)** |
| Random access | `deque[i]` | Середина | **O(n)** ← компроміс |

> **Компроміс:** `deque` швидкий на кінцях, але повільний для індексного доступу до середини.  
> Якщо потрібно часто читати `data[i]` — використовуйте `list`.

In [ ]:
from collections import deque

# === ВСІ БАЗОВІ ОПЕРАЦІЇ deque ===

d = deque([10, 20, 30])
print(f"Початковий deque: {d}")

# Додаємо з обох кінців
d.append(40)       # → правий кінець
d.appendleft(0)    # → лівий кінець
print(f"Після append(40) та appendleft(0): {d}")

# Вилучаємо з обох кінців
right = d.pop()         # Знімаємо з правого
left  = d.popleft()     # Знімаємо з лівого
print(f"Після pop() та popleft(): {d}")
print(f"  Вилучено праворуч: {right}, ліворуч: {left}")

print()

# rotate — циклічний зсув
carousel = deque(["A", "B", "C", "D", "E"])
print(f"До rotate(2)  : {carousel}")
carousel.rotate(2)   # Зсуває праворуч на 2 позиції
print(f"Після rotate(2): {carousel}")
carousel.rotate(-1)  # Зсуває ліворуч на 1 позицію
print(f"Після rotate(-1): {carousel}")

### Патерн: Circular Buffer (Кільцевий буфер) з `maxlen`

Один із найпотужніших, але маловідомих параметрів `deque` — **`maxlen`**.  
Якщо вказати максимальну довжину, `deque` автоматично перетворюється на **кільцевий буфер**:

- При додаванні нового елемента у повний буфер **найстаріший елемент автоматично видаляється** з протилежного кінця.
- Це O(1) — не потрібен жодний явний код видалення.

**Застосування:**
- Останні N рядків лог-файлу
- Ковзне середнє (moving average) для фінансових даних
- Буфер останніх N сторінок у браузері

In [ ]:
from collections import deque

# === CIRCULAR BUFFER: останні 5 рядків логу ===
LOG_BUFFER_SIZE = 5
log_tail = deque(maxlen=LOG_BUFFER_SIZE)  # Кільцевий буфер на 5 рядків

log_lines = [
    "INFO: Сервер запущений",
    "INFO: З'єднання з БД встановлено",
    "WARNING: Повільний запит (2.3s)",
    "ERROR: Диск заповнений на 95%",
    "INFO: Автоматичне очищення запущено",
    "ERROR: Таймаут підключення до БД",
    "INFO: Повторна спроба підключення",
    "INFO: З'єднання відновлено",
]

for line in log_lines:
    log_tail.append(line)  # Якщо буфер повний — найстаріший рядок зникає автоматично

print(f"Всього рядків у логу: {len(log_lines)}")
print(f"Збережено останніх {LOG_BUFFER_SIZE}:")
for i, line in enumerate(log_tail, 1):
    print(f"  {i}. {line}")

print()

# === MOVING AVERAGE: ковзне середнє цін ===
WINDOW = 3
price_window = deque(maxlen=WINDOW)

prices = [100, 102, 98, 105, 103, 110, 108]

print(f"Ковзне середнє (вікно={WINDOW}):")
for price in prices:
    price_window.append(price)
    if len(price_window) == WINDOW:  # Рахуємо лише коли вікно заповнене
        avg = sum(price_window) / WINDOW
        print(f"  {list(price_window)} → avg = {avg:.2f}")

---

## Частина 4: `queue.Queue` — Thread-Safe Черга для Конкурентності

---

### `collections.deque` vs `queue.Queue`

Важливо розрізняти ці два інструменти:

| | `collections.deque` | `queue.Queue` |
|---|---|---|
| **Призначення** | Структура даних | Комунікаційний канал між потоками |
| **Thread-safe** | Атомарні pop/append — так | Повністю безпечна — так |
| **Блокування** | Немає | `get()` блокує потік до появи даних |
| **Overhead** | Мінімальний (C) | Mutex locks, більший overhead |
| **Коли використовувати** | Алгоритми, одно-потокові структури | Передача задач між потоками |

> **Правило:** `collections.deque` для «сирих» структур даних.  
> `queue.Queue` — виключно для передачі повідомлень між concurrent потоками/процесами.

In [ ]:
import queue
import threading
import time

# === PRODUCER-CONSUMER pattern з queue.Queue ===

task_queue = queue.Queue(maxsize=3)  # Буфер максимум на 3 задачі

def producer(n_tasks: int) -> None:
    """Генерує задачі і кладе в чергу."""
    for i in range(n_tasks):
        task = f"Task-{i+1:02d}"
        task_queue.put(task)          # Блокує, якщо черга заповнена (maxsize)
        print(f"  [Producer] Додано: {task}")
        time.sleep(0.05)

def consumer(worker_id: int) -> None:
    """Бере задачі з черги і обробляє їх."""
    while True:
        try:
            task = task_queue.get(timeout=0.5)  # Чекає 0.5с, потім виходить
            print(f"  [Worker-{worker_id}] Обробка: {task}")
            time.sleep(0.1)           # Імітуємо обробку
            task_queue.task_done()    # Сигналізуємо про завершення
        except queue.Empty:
            break                     # Черга порожня і timeout — виходимо

# Запускаємо 1 producer і 2 consumers
producer_thread = threading.Thread(target=producer, args=(6,))
worker_threads  = [threading.Thread(target=consumer, args=(i,)) for i in range(1, 3)]

producer_thread.start()
for t in worker_threads:
    t.start()

producer_thread.join()
for t in worker_threads:
    t.join()

print("\nВсі задачі оброблено.")

---

## Частина 5: Explicit Stack — DFS без `RecursionError`

---

### Рекурсія vs Explicit Stack

Рекурсивний DFS зручний для читання, але має критичне обмеження:  
**Python's call stack** обмежений ~1000 фреймів. Глибокі дерева → `RecursionError`.

**Архітектурне рішення:** замінити call stack на **явний список** (`list` як стек).  
Список розміщується в heap (купі пам'яті) — обмежений лише RAM системи.

| | Рекурсивний DFS | Iterative DFS (explicit stack) |
|---|---|---|
| **Обмеження глибини** | ~1000 фреймів | Лише RAM |
| **Читаємість** | Вища | Нижча |
| **Overhead** | Новий stack frame на кожен виклик | Тільки `list.append/pop` |
| **Коли використовувати** | Невелика глибина (<500) | Глибокі структури, продакшн |

In [ ]:
from typing import Iterator

# Граф: файлова система як дерево директорій
filesystem = {
    "/": ["/home", "/etc", "/var"],
    "/home": ["/home/alice", "/home/bob"],
    "/home/alice": ["/home/alice/docs", "/home/alice/code"],
    "/home/alice/docs": [],
    "/home/alice/code": ["/home/alice/code/project"],
    "/home/alice/code/project": [],
    "/home/bob": [],
    "/etc": ["/etc/nginx"],
    "/etc/nginx": [],
    "/var": ["/var/log"],
    "/var/log": [],
}

def dfs_recursive(graph: dict, node: str, depth: int = 0) -> None:
    """DFS рекурсивний — зручно, але обмежений ~1000 рівнями."""
    print("  " + "  " * depth + f"📁 {node}")
    for neighbor in graph.get(node, []):
        dfs_recursive(graph, neighbor, depth + 1)  # Push на call stack


def dfs_iterative(graph: dict, start: str) -> Iterator[str]:
    """DFS ітеративний — явний стек, необмежена глибина."""
    stack = [(start, 0)]      # Кожен елемент: (вузол, рівень вкладеності)
    visited = set()
    
    while stack:
        node, depth = stack.pop()    # Pop з кінця списку (O(1))
        if node in visited:
            continue
        visited.add(node)
        yield node, depth
        
        # Push сусідів у зворотньому порядку — щоб обходити зліва направо
        for neighbor in reversed(graph.get(node, [])):
            if neighbor not in visited:
                stack.append((neighbor, depth + 1))


print("=== Рекурсивний DFS ===")
dfs_recursive(filesystem, "/")

print()
print("=== Ітеративний DFS (explicit stack) ===")
for node, depth in dfs_iterative(filesystem, "/"):
    print("  " + "  " * depth + f"📁 {node}")

### DFS vs BFS: Порівняльна демонстрація

Побачимо наочну різницю в порядку обходу між стеком (DFS) і чергою (BFS):

In [ ]:
from collections import deque

# Простий граф для порівняння
graph = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": [], "E": [], "F": [], "G": [],
}

def dfs(graph: dict, start: str) -> list[str]:
    """DFS через explicit stack (list) — занурюється спочатку."""
    stack, visited, order = [start], set(), []
    while stack:
        node = stack.pop()           # LIFO: останній push → перший pop
        if node not in visited:
            visited.add(node)
            order.append(node)
            for n in reversed(graph.get(node, [])):
                stack.append(n)
    return order

def bfs(graph: dict, start: str) -> list[str]:
    """BFS через deque (queue) — рівень за рівнем."""
    queue, visited, order = deque([start]), {start}, []
    while queue:
        node = queue.popleft()       # FIFO: перший push → перший pop
        order.append(node)
        for n in graph.get(node, []):
            if n not in visited:
                visited.add(n)
                queue.append(n)
    return order


dfs_order = dfs(graph, "A")
bfs_order = bfs(graph, "A")

print("Граф:")
print("       A")
print("      / \\")
print("     B   C")
print("    / \\ / \\")
print("   D  E F  G")
print()
print(f"DFS (Stack/LIFO) порядок: {' → '.join(dfs_order)}")
print(f"BFS (Queue/FIFO) порядок: {' → '.join(bfs_order)}")
print()
print("DFS занурюється: A→B→D (ліва гілка до кінця), потім відходить")
print("BFS розширюється: A (рівень 0), B,C (рівень 1), D,E,F,G (рівень 2)")

---

## Архітектурний Підсумок

---

| Структура | Тип потоку | Python-клас | Push | Pop | Застосування |
|-----------|-----------|-------------|------|-----|-------------|
| Stack | LIFO | `list` | `append()` | `pop()` | Рекурсія, DFS, Undo, парсери |
| Queue | FIFO | `deque` | `append()` | `popleft()` | BFS, Task Queue, Spooling |
| Deque | Двосторонній | `deque` | `append/left` | `pop/left` | Sliding Window, Ring Buffer |
| Thread Queue | FIFO + locks | `queue.Queue` | `put()` | `get()` | Producer-Consumer |
| Priority Queue | За пріоритетом | `heapq` | `heappush()` | `heappop()` | A*, Dijkstra, Scheduler |

> **Ключова думка:**  
> Вибір структури даних — це вибір **політики виконання**.  
> Stack каже: «Найновіше важливіше». Queue каже: «Хто прийшов першим — має право першим».  
> Ваш алгоритм є прямим наслідком цього вибору.

---

## Питання для Співбесіди

---

### 1. Чому `list.pop(0)` є поганою практикою для реалізації черги?

> **Відповідь:**  
> `list` у Python — це динамічний масив (contiguous block of memory).  
> `pop(0)` видаляє перший елемент і змушує інтерпретатор **зсунути всі наступні** n-1 елементів на одну позицію вліво.  
> Це операція O(n) — при 1 мільйоні елементів кожна операція виконує мільйон зсувів.  
> У циклі обробки черги це деградує до **O(n²)** — катастрофа для продуктивності.  
> Правильне рішення: `collections.deque.popleft()` — O(1), бо `deque` — це doubly-linked list з вказівниками на обидва кінці.

---

### 2. Коли BFS гарантовано знаходить найкоротший шлях, а коли — ні?

> **Відповідь:**  
> BFS **гарантує найкоротший шлях** (за кількістю ребер) у **незважених графах**, де всі ребра мають рівну «вартість» переходу (або вагу = 1).  
> Причина: FIFO-черга забезпечує обробку вузлів строго у порядку зростання відстані від старту — спочатку всі вузли на відстані 1, потім на відстані 2, тощо.  
> У **зважених графах** (де ребра мають різну вагу) BFS не гарантує найкоротший шлях.  
> Для зважених графів потрібен **алгоритм Дейкстри** (з `heapq` — priority queue).

---

### 3. Чим відрізняється `collections.deque` від `queue.Queue`? Коли використовувати кожен?

> **Відповідь:**  
> `collections.deque` — це **структура даних**: швидка, мінімальний overhead, реалізована на C.  
> Для атомарних `append/popleft` — thread-safe, але **не має блокувань** — потоки не чекатимуть на дані.  
> `queue.Queue` — це **комунікаційний канал між потоками**: обгорнутий mutex locks, `get()` блокує потік до появи нових даних.  
> **Правило:** `deque` для алгоритмів і single-threaded структур; `queue.Queue` виключно для координації concurrent producers і consumers.

---

### 4. Що таке кільцевий буфер і як реалізувати його в одну строчку Python?

> **Відповідь:**  
> Кільцевий буфер (Circular Buffer / Ring Buffer) — структура фіксованого розміру.  
> При надходженні нового елемента в повний буфер **найстаріший елемент автоматично видаляється**.  
> В Python: `deque(maxlen=N)`. Приклад: `log = deque(maxlen=100)` — завжди зберігає останні 100 рядків логу, без жодного явного коду видалення. Всі операції O(1).

---

### 5. Яка архітектурна перевага iterative DFS над recursive DFS?

> **Відповідь:**  
> Рекурсивний DFS використовує **call stack** CPython — обмежений ~1000 фреймів (`sys.getrecursionlimit()`).  
> При перевищенні — `RecursionError`. Для глибоких дерев (файлові системи, великі графи) це критично.  
> Iterative DFS замінює call stack на **явний `list`** (heap memory) — обмежений лише RAM.  
> Додаткова перевага: менший overhead (немає створення нових stack frames на кожен виклик).

---

## Додаткові Завдання для Студентів

---

### Завдання 1 — Undo/Redo (Рівень: Середній)

Реалізуйте клас `TextEditor` з підтримкою операцій `write(text)`, `undo()` і `redo()`.  
Вимоги:
- `write(text)` — додає текст до документа, очищає redo-стек
- `undo()` — скасовує останню операцію запису, переміщує дію в redo-стек
- `redo()` — повторно застосовує скасовану операцію
- `get_content()` — повертає поточний вміст документа

**Підказка:** використайте два стеки: `history` і `redo_stack`.

In [ ]:
# === Завдання 1: Undo/Redo ===
# Реалізуйте клас TextEditor з двома стеками

class TextEditor:
    def __init__(self):
        # TODO: ініціалізуйте два стеки — history та redo_stack
        pass
    
    def write(self, text: str) -> None:
        # TODO: збережіть поточний стан у history, додайте текст, очистіть redo_stack
        pass
    
    def undo(self) -> None:
        # TODO: перемістіть поточний стан у redo_stack, відновіть попередній
        pass
    
    def redo(self) -> None:
        # TODO: перемістіть стан з redo_stack назад у history
        pass
    
    def get_content(self) -> str:
        # TODO: повертає поточний вміст
        pass


# Тести
editor = TextEditor()
editor.write("Hello")
editor.write(" World")
print(editor.get_content())  # Очікується: "Hello World"
editor.undo()
print(editor.get_content())  # Очікується: "Hello"
editor.redo()
print(editor.get_content())  # Очікується: "Hello World"

### Завдання 2 — Hot Potato (Рівень: Середній)

Реалізуйте симуляцію гри «Гаряча картопля» (Hot Potato) за допомогою `deque`.  
Правила:
- Є список учасників, що стоять у колі
- Кожен раунд: передаємо «картоплю» `n` разів по колу
- Учасник, який тримає картоплю після `n` передач, вибуває
- Переможець — останній учасник

**Підказка:** використайте `deque.rotate(-1)` або `appendleft(popleft())`.

In [ ]:
from collections import deque

# === Завдання 2: Hot Potato ===

def hot_potato(players: list[str], num_passes: int) -> str:
    """
    Симуляція гри Hot Potato.
    :param players: список учасників
    :param num_passes: кількість передач картоплі в кожному раунді
    :return: ім'я переможця
    """
    circle = deque(players)
    
    # TODO: реалізуйте логіку гри
    # Підказка: rotate(-num_passes) переміщує "картоплю" на num_passes позицій
    # Потім popleft() вилучає того, хто вибуває
    pass


# Тест
players = ["Alice", "Bob", "Carol", "Dave", "Eve"]
winner = hot_potato(players, num_passes=7)
print(f"Переможець: {winner}")

### Завдання 3 — Sliding Window Maximum (Рівень: Складний)

Дано масив чисел і розмір вікна `k`. Знайдіть максимальне значення в кожному ковзному вікні.

**Приклад:** `nums=[1,3,-1,-3,5,3,6,7]`, `k=3` → `[3,3,5,5,6,7]`

**Наївне рішення:** O(n·k) — дві вкладені петлі.  
**Оптимальне рішення:** O(n) — використайте `deque` для зберігання індексів потенційних максимумів.

**Підказка:** `deque` зберігає індекси в **спадному порядку** значень.  
Коли нове число більше за елемент у хвості — видаляємо хвіст (`pop()`).  
Коли голова вийшла за межі вікна — видаляємо голову (`popleft()`).

In [ ]:
from collections import deque

# === Завдання 3: Sliding Window Maximum ===

def sliding_window_max(nums: list[int], k: int) -> list[int]:
    """
    Знаходить максимум у кожному вікні розміром k за O(n).
    :param nums: масив чисел
    :param k: розмір вікна
    :return: список максимумів для кожного вікна
    """
    # TODO: реалізуйте рішення через deque
    # deque зберігає ІНДЕКСИ (не значення) у спадному порядку значень
    pass


# Тести
nums = [1, 3, -1, -3, 5, 3, 6, 7]
result = sliding_window_max(nums, k=3)
print(f"nums = {nums}")
print(f"k = 3")
print(f"Максимуми вікон: {result}")
print(f"Очікується:     [3, 3, 5, 5, 6, 7]")
print(f"Правильно: {result == [3, 3, 5, 5, 6, 7]}")

  ---
  cd module_3/lessons/lesson_23_deque_queue/queue_policy_lab

  # App 1 — Live Dispatch Room (порт 8551)
  python run_dispatcher.py

  # App 2 — Policy Analysis Lab (порт 8552)
  python run_policy_lab.py
